# 6 WorkFlow Gerencial, futuro=SEP — versión Google Cloud / Jupyter


## Nota de trazabilidad

Esta es una **versión limpia de la configuración de referencia final 7196** usada para la comparación de experimentos: semilla `895651` y `training_pct = 1.0` (100% de CONTINUA durante el Grid Search).  
Se eliminaron outputs heredados de una corrida anterior al 40% para evitar mezclar resultados de configuraciones distintas.


> **VERSIÓN FINAL LIMPIA 7196 — VÍCTOR**
>
> Esta versión conserva el workflow validado de la corrida **7195**:
> FE intra-mes (#03 Grupo B) + GA con `gramEvol` (#10 Grupo A) +
> FE histórico (#05 Grupo B) + undersampling 10% (#12) +
> Grid Search + Final Training.
>
> Está preparada para ejecutarse completa con **Run All**.
>
> **Seguridad Kaggle:**
> 1. Al comienzo hace un **control de acceso** a la competencia real
>    `utn-2026-virtual-mgr`. No hace submit.
> 2. Al final genera los 11 CSV con el envío **DESHABILITADO**.
>    Para subirlos hay que cambiar explícitamente una sola línea
>    de `FALSE` a `TRUE` y volver a ejecutar **solo la celda PASO 2**.
>
> Un `Run All` completo nunca envía submissions por accidente.


### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2 Seteo del ambiente en Google Cloud / Jupyter

Esta variante conserva el workflow Gerencial original y cambia únicamente la preparación del ambiente.

- No monta Google Drive.
- Trabaja directamente sobre el disco persistente de la VM.
- Se ejecuta con kernel **R** desde el inicio.
- Las rutas se concentran en variables para evitar referencias a `/content/...` propias de Colab.


In [ ]:
# ============================================================
# RUTAS DE TRABAJO - GOOGLE CLOUD / JUPYTER
# ============================================================
# Si la VM usa otra ubicación, alcanza con cambiar estas variables
# o definir DM_BUCKET / DM_DATASETS_DIR / DM_EXP_DIR en el entorno.

BASE_BUCKET <- Sys.getenv("DM_BUCKET", unset = "/home/ds/buckets/b1")
DATASETS_DIR <- Sys.getenv("DM_DATASETS_DIR", unset = file.path(BASE_BUCKET, "datasets"))
EXP_DIR <- Sys.getenv("DM_EXP_DIR", unset = file.path(BASE_BUCKET, "exp"))

dir.create(DATASETS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)

cat("BASE_BUCKET :", BASE_BUCKET, "\n")
cat("DATASETS_DIR:", DATASETS_DIR, "\n")
cat("EXP_DIR     :", EXP_DIR, "\n")


#### Descarga de datasets si todavía no existen en la VM

El notebook original los copia entre Google Drive y `/content`. En Cloud no hace falta duplicarlos: se guardan directamente en `DATASETS_DIR`.


In [ ]:
# Descarga solamente si el archivo todavía no está presente
URL_ORIGEN <- "https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"

descargar_si_falta <- function(archivo) {
  destino <- file.path(DATASETS_DIR, archivo)
  if (!file.exists(destino)) {
    cat("Descargando", archivo, "...\n")
    download.file(paste0(URL_ORIGEN, archivo), destino, mode = "wb", quiet = FALSE)
  } else {
    cat("Ya existe:", destino, "\n")
  }
}

descargar_si_falta("dataset_pequeno.csv")
descargar_si_falta("gerencial_competencia_2026.csv.gz")


#### PASO 1 — Control de Kaggle (NO envía nada)

Este control se ejecuta al principio del notebook.

Verifica:

- que exista la CLI de Kaggle;
- que exista `~/.kaggle/kaggle.json`;
- que la cuenta configurada tenga acceso a la competencia real
  **`utn-2026-virtual-mgr`**.

Si termina con `CONTROL KAGGLE OK`, se puede dejar correr el resto del notebook.
**Esta celda no hace ningún submit.**


In [ ]:
# ================================================================
# PASO 1 - CONTROL DE ACCESO A KAGGLE
# SOLO LECTURA: NO HACE SUBMIT
# ================================================================

competencia_control <- "utn-2026-virtual-mgr"

kaggle_cmd <- Sys.which("kaggle")
kaggle_json <- path.expand("~/.kaggle/kaggle.json")

cat("Kaggle CLI :", ifelse(nchar(kaggle_cmd) > 0, kaggle_cmd, "NO ENCONTRADO"), "\n")
cat("kaggle.json:", ifelse(file.exists(kaggle_json), kaggle_json, "NO ENCONTRADO"), "\n")
cat("Competencia:", competencia_control, "\n")

if (nchar(kaggle_cmd) == 0)
  stop("CONTROL KAGGLE: no se encontró el comando kaggle en el PATH.")

if (!file.exists(kaggle_json))
  stop("CONTROL KAGGLE: no se encontró ~/.kaggle/kaggle.json.")

salida_control <- suppressWarnings(
  system2(
    kaggle_cmd,
    args = c(
      "competitions", "submissions",
      "-c", competencia_control,
      "--page-size", "1",
      "--csv",
      "--quiet"
    ),
    stdout = TRUE,
    stderr = TRUE
  )
)

estado_control <- attr(salida_control, "status")
if (is.null(estado_control)) estado_control <- 0

if (estado_control != 0) {
  cat(paste(salida_control, collapse = "\n"), "\n")
  stop(
    "CONTROL KAGGLE: no se pudo acceder a la competencia ",
    competencia_control,
    ". Revisar cuenta, token o acceso."
  )
}

cat("\n============================================================\n")
cat("CONTROL KAGGLE OK\n")
cat("Acceso confirmado a:", competencia_control, "\n")
cat("NO se realizó ningún submit.\n")
cat("============================================================\n")


## 6.3  Workflow

## Inicializacion

Esta versión Cloud/Jupyter se ejecuta con kernel **R** desde el comienzo. No es necesario cambiar de runtime.


limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")


In [ ]:
# limpio la memoria, PERO conservo las rutas de trabajo de Cloud
# porque fueron definidas antes y se usan más adelante.
objetos_a_conservar <- c("BASE_BUCKET", "DATASETS_DIR", "EXP_DIR")

rm(
  list = setdiff(
    ls(all.names = TRUE),
    objetos_a_conservar
  )
)

gc(full = TRUE, verbose = FALSE) # garbage collection

# control: las rutas deben seguir existiendo después de limpiar memoria
stopifnot(
  exists("BASE_BUCKET"),
  exists("DATASETS_DIR"),
  exists("EXP_DIR")
)

cat("Rutas conservadas después de limpiar memoria:\n")
cat("BASE_BUCKET :", BASE_BUCKET, "\n")
cat("DATASETS_DIR:", DATASETS_DIR, "\n")
cat("EXP_DIR     :", EXP_DIR, "\n")


In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")


#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 895651

PARAM$experimento <- 7196
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"


# ================================================================
# PROBLEMA #10 - GRUPO A
# Feature Engineering intra-mes mediante Algoritmo Genético (gramEvol)
# Adaptado desde 620_WorkFlow_01_junior_exp10_grupoA.ipynb
# ================================================================
PARAM$GA <- list(
  popSize = 150,               # población
  iterations = 50,             # generaciones
  top_features = 20,           # features GA a inyectar
  max_terminales = 60,         # máximo de variables terminales
  seqLen = 250,                # longitud máxima del genoma
  max.depth = 10,              # profundidad máxima BNF
  max_filas_fitness = 50000,   # muestra máxima para fitness

  # Valores que realmente ejecuta el notebook del Grupo A
  mutationChance = 0.15,
  elitism_fraction = 0.50
)


# ================================================================
# MODO CONTROL
# FALSE = modo seguro para ejecutar todo con Run All
# NO cambiar aquí. El envío se habilita únicamente en el PASO 2 final.
# ================================================================
PARAM$control <- list()
PARAM$control$submit_kaggle <- TRUE



#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd(EXP_DIR)
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd(file.path(EXP_DIR, experimento_folder))

# dejo visible la carpeta efectiva del experimento
cat("Experimento:", getwd(), "\n")


### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
archivo_dataset <- file.path(DATASETS_DIR, PARAM$dataset)
stopifnot(file.exists(archivo_dataset))
dataset <- fread(archivo_dataset)
cat("Dataset:", archivo_dataset, "\n")
cat("Filas:", nrow(dataset), " Columnas:", ncol(dataset), "\n")


#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]


#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow


#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# ================================================================
# 6.3.1.3 FE_intra_manual
# Recomendación elegida: PROBLEMA #03 - GRUPO B - EXPERIMENTO 2
# Flags + Z-Score + ratios no redundantes
# Adaptado al z719 final Gerencial - futuro 202109
# ================================================================


# Verifica que existan todas las variables necesarias
atributos_presentes <- function(patributos)
{
  atributos <- unique(patributos)
  comun <- intersect(atributos, colnames(dataset))

  return(length(atributos) == length(comun))
}


# ------------------------------------------------
# División segura
# Si el denominador es 0, lo convierte en NA
# ------------------------------------------------
divseg <- function(numerador, denominador)
{
  denominador[denominador == 0] <- NA
  return(numerador / denominador)
}


# ================================================================
# VARIABLES ORIGINALES DEL z719
# ================================================================

# Mes: 1, 2, ..., 12
if(atributos_presentes(c("foto_mes")))
  dataset[, kmes := foto_mes %% 100]


# Variable original del workflow
if(atributos_presentes(c("mpayroll", "cliente_edad")))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


# ================================================================
# EXPERIMENTO 2 - GRUPO B
# ================================================================

# 10 variables utilizadas por el Grupo B
vars_top10 <- c(
  "ctrx_quarter",
  "mcaja_ahorro",
  "cpayroll_trx",
  "mcuentas_saldo",
  "mcuenta_corriente",
  "mprestamos_personales",
  "mtarjeta_visa_consumo",
  "Visa_mpagominimo",
  "cliente_edad",
  "mpasivos_margen"
)


# Variables que pueden presentar ceros por no tener el producto
vars_alto_riesgo <- c(
  "cpayroll_trx",
  "mcuenta_corriente",
  "mprestamos_personales",
  "Visa_mpagominimo"
)


# ================================================================
# A) FLAGS DE TENENCIA
# ================================================================

if(atributos_presentes(vars_alto_riesgo))
{
  for(v in vars_alto_riesgo)
  {
    nombre_flag <- paste0("flag_tiene_", v)

    dataset[, (nombre_flag) := as.integer(get(v) > 0)]
  }
}


# ================================================================
# B) Z-SCORE
#
# IMPORTANTE:
# Grupo B calculaba media y desvío SOLO con su training.
#
# Como nuestro z719 final usa:
# training   = 202005 ... 202106
# validation = 202107
# future     = 202109
#
# calculamos media y SD SOLO con 202005 ... 202106.
# Así evitamos usar información de validation/future.
# ================================================================

if(atributos_presentes(vars_top10))
{
  filas_estandarizacion <- dataset[
    foto_mes %in% c(
      202005, 202006, 202007, 202008, 202009, 202010,
      202011, 202012,
      202101, 202102, 202103, 202104, 202105, 202106
    )
  ]

  for(v in vars_top10)
  {
    # Media calculada solamente con training
    media_v <- mean(
      filas_estandarizacion[[v]],
      na.rm = TRUE
    )

    # Desvío estándar calculado solamente con training
    desvio_v <- sd(
      filas_estandarizacion[[v]],
      na.rm = TRUE
    )

    # Nombre de la nueva variable
    nombre_z <- paste0("z_", v)

    # Creo Z-Score solo si el desvío es válido
    if(is.finite(desvio_v) && desvio_v > 0)
    {
      dataset[, (nombre_z) := (get(v) - media_v) / desvio_v]
    }
    else
    {
      dataset[, (nombre_z) := NA_real_]
    }

    # Muestro los parámetros utilizados
    cat(sprintf(
      "Estandarizada %s | media training = %.6f | sd training = %.6f\n",
      v, media_v, desvio_v
    ))
  }

  # Libero tabla temporal
  rm(filas_estandarizacion)
}


# ================================================================
# C) RATIOS NO REDUNDANTES
#
# 10 variables => 45 pares únicos
# Se genera A/B, pero NO además B/A.
# ================================================================

vars_z_top10 <- paste0("z_", vars_top10)

if(atributos_presentes(vars_z_top10))
{
  n_vars <- length(vars_z_top10)

  for(i in 1:(n_vars - 1))
  {
    for(j in (i + 1):n_vars)
    {
      num <- vars_z_top10[i]
      den <- vars_z_top10[j]

      nombre_nueva <- paste0(
        "ratio_",
        vars_top10[i],
        "_sobre_",
        vars_top10[j],
        "_z"
      )

      dataset[, (nombre_nueva) := divseg(
        get(num),
        get(den)
      )]
    }
  }
}


In [ ]:
# ================================================================
# CONTROL 1 - FE INTRA-MES
# No modifica datos: solamente verifica que se hayan creado
# las variables esperadas del Experimento 2 del Grupo B.
# ================================================================

flags_esperados <- paste0("flag_tiene_", vars_alto_riesgo)
z_esperados <- paste0("z_", vars_top10)

faltan_flags <- setdiff(flags_esperados, colnames(dataset))
faltan_z <- setdiff(z_esperados, colnames(dataset))
ratios_creados <- grep("^ratio_.*_z$", colnames(dataset), value = TRUE)

if (length(faltan_flags) > 0)
  stop("CONTROL FE intra-mes: faltan flags: ", paste(faltan_flags, collapse = ", "))

if (length(faltan_z) > 0)
  stop("CONTROL FE intra-mes: faltan variables Z-Score: ", paste(faltan_z, collapse = ", "))

if (length(ratios_creados) < 45)
  stop("CONTROL FE intra-mes: se esperaban al menos 45 ratios y se encontraron ", length(ratios_creados))

cat(
  "CONTROL FE intra-mes OK | filas:", nrow(dataset),
  "| columnas:", ncol(dataset),
  "| flags:", length(flags_esperados),
  "| z-score:", length(z_esperados),
  "| ratios:", length(ratios_creados), "\n"
)


#### 6.3.1.3.2 FE_intra_GA — Problema #10, Grupo A

**Decisión elegida:** incorporar Feature Engineering intra-mes mediante
Algoritmo Genético con `gramEvol`, adaptando el bloque del notebook
`620_WorkFlow_01_junior_exp10_grupoA.ipynb` al `z719` Gerencial.

Este bloque se ejecuta después del FE intra-mes del Problema #03 y antes
de FEhist, para que las variables `GA_Feature_*` también puedan recibir
lags, deltas y media móvil en el bloque histórico.

**Importante:** se conserva la lógica ejecutable del notebook del Grupo A.
La población, generaciones, top features, terminales, `seqLen` y `max.depth`
son los del archivo fuente. Los parámetros de presión selectiva que realmente
ejecuta ese notebook son `mutationChance = 0.15` y `elitism = 50%` de la población.


In [ ]:
# ==============================================================================
# 6.3.1.3.2 FE_intra_GA: Algoritmo Genético (Grammatical Evolution)
# ==============================================================================

if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cloud.r-project.org", dependencies = TRUE)
}
if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cran.rstudio.com", dependencies = TRUE)
}
require("gramEvol")

if (!require("lightgbm")) {
  install.packages("lightgbm", repos = "https://cloud.r-project.org")
}
require("lightgbm")
require("data.table")
require("parallel")

if (is.null(PARAM$GA)) {
  PARAM$GA <- list(
    popSize = 150,            
    iterations = 50,          
    top_features = 20,        
    max_terminales = 60,      
    seqLen = 250,             
    max.depth = 10,           
    max_filas_fitness = 50000 
  )
}

# carpeta efectiva del experimento actual (WF7193)
dir_experimento_base <- getwd()

# 1. Variables candidatas para el Algoritmo Genético
excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# Filtrar únicamente variables numéricas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# Excluir fechas
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# Límite de seguridad de terminales para controlar el espacio de búsqueda
MAX_TERMINALES <- if (!is.null(PARAM$GA$max_terminales)) PARAM$GA$max_terminales else 60
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para Grammatical Evolution:", length(candidatas_GA), "\n")

# 2. Split Local de Validación (sin tocar 202107 ni 202109)
meses_disponibles <- sort(unique(dataset$foto_mes[dataset$foto_mes < 202107]))
meses_val_GA <- tail(meses_disponibles, 3)
meses_tr_GA  <- setdiff(meses_disponibles, meses_val_GA)

idx_tr_all <- which(dataset$foto_mes %in% meses_tr_GA)
idx_val_all <- which(dataset$foto_mes %in% meses_val_GA)

# Subsampling controlado para evaluación de fitness ultrarrápida
set.seed(PARAM$semilla_primigenia)
n_fit <- if (!is.null(PARAM$GA$max_filas_fitness)) PARAM$GA$max_filas_fitness else 50000
idx_tr_GA <- if (length(idx_tr_all) > n_fit) sample(idx_tr_all, n_fit) else idx_tr_all
idx_val_GA <- if (length(idx_val_all) > (n_fit / 2)) sample(idx_val_all, n_fit / 2) else idx_val_all

y_tr_GA <- ifelse(dataset$clase_ternaria[idx_tr_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)
y_val_GA <- ifelse(dataset$clase_ternaria[idx_val_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)

# 3. Operadores Protegidos
protected_div <- function(x, y) {
  res <- x / (y + 1e-5)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

protected_log_diff <- function(x, y) {
  res <- log(abs(x - y) + 1)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

es_expresion_trivial <- function(f) {
  !grepl("[+*/-]|protected_", f)
}

# 4. Definición de la Gramática BNF
string_vars <- paste(candidatas_GA, collapse = " | ")
rule_text <- paste0(
  "<expr> ::= <op>\n",
  "<op>   ::= <op> + <op> | <op> - <op> | <op> * <op> | ",
  "protected_div(<op>, <op>) | protected_log_diff(<op>, <op>) | <var>\n",
  "<var>  ::= ", string_vars
)

tf <- tempfile()
writeLines(rule_text, tf)
bnf_grammar <- CreateGrammar(tf)
unlink(tf)

# 5. Función de Fitness con LightGBM Univariado Real
fitness_gramEvol <- function(expr) {
  valores <- tryCatch(eval(expr, envir = dataset), error = function(e) NULL)
  if (is.null(valores)) return(1)

  val_tr <- valores[idx_tr_GA]
  val_finitos <- val_tr[is.finite(val_tr)]
  if (length(val_finitos) == 0 || length(unique(val_finitos)) <= 1) {
    return(1)
  }

  dtr_ga  <- lgb.Dataset(data = matrix(val_tr, ncol = 1), label = y_tr_GA, free_raw_data = TRUE)
  dval_ga <- lgb.Dataset(data = matrix(valores[idx_val_GA], ncol = 1), label = y_val_GA, free_raw_data = TRUE)

  modelo_ga <- tryCatch({
    lgb.train(
      params = list(objective = "binary", metric = "auc",
                    learning_rate = 0.1, num_threads = 1, verbosity = -1),
      data = dtr_ga, valids = list(valid = dval_ga),
      nrounds = 50, early_stopping_rounds = 10, verbose = -1
    )
  }, error = function(e) NULL)

  if (is.null(modelo_ga) || is.null(modelo_ga$best_score) || is.na(modelo_ga$best_score)) return(1)

  1 - modelo_ga$best_score
}

evaluar_genoma <- function(genoma) {
  expr_obj <- tryCatch(suppressWarnings(GrammarMap(genoma, bnf_grammar)), error = function(e) NULL)
  if (is.null(expr_obj) || !isTRUE(GrammarIsTerminal(expr_obj))) {
    return(list(score = 0, formula = NA_character_))
  }
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) {
    return(list(score = 0, formula = NA_character_))
  }

  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final, width.cutoff = 500L), collapse = " ")
  costo <- fitness_gramEvol(expr_final)
  auc   <- 1 - costo

  list(score = auc, formula = formula_str)
}

# 6. Ejecución del Algoritmo Genético
set.seed(PARAM$semilla_primigenia)

cat("\n===================================================================\n")
cat(">>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol) <<<\n")
cat("===================================================================\n")

ge_res <- GrammaticalEvolution(
  grammarDef      = bnf_grammar,
  evalFunc        = fitness_gramEvol,
  popSize         = PARAM$GA$popSize,
  iterations      = PARAM$GA$iterations,
  terminationCost = 0.10,
  seqLen          = PARAM$GA$seqLen,
  max.depth       = PARAM$GA$max.depth,
  
  # Nuevos parámetros de presión selectiva
  elitism         = as.integer(PARAM$GA$popSize * PARAM$GA$elitism_fraction),
  mutationChance  = PARAM$GA$mutationChance,
  
  monitorFunc     = function(result) {
    cat(sprintf("Gen %2d | Mejor Costo: %.5f (AUC: %.5f)\n",
                result$population$currentIteration,
                result$best$cost,
                1 - result$best$cost))
  }
)

# 7. Extracción e Inyección del Top N al Dataset
cat("\n=== Evaluando población final para extraer el Top", PARAM$GA$top_features, "===\n")

pop_matrix      <- ge_res$population$population
poblacion_final <- split(pop_matrix, row(pop_matrix))

n_cores <- max(1, detectCores() - 1)
resultados <- if (.Platform$OS.type == "unix") {
  mclapply(poblacion_final, evaluar_genoma, mc.cores = n_cores)
} else {
  lapply(poblacion_final, evaluar_genoma)
}

scores_finales   <- sapply(resultados, function(r) r$score)
formulas_finales <- sapply(resultados, function(r) r$formula)

ordenados <- order(scores_finales, decreasing = TRUE)

formulas_vistas <- character()
ga_cols_creadas <- character()
top_guardados   <- 0
idx             <- 1

dt_trazabilidad <- data.table(Variable=character(), AUC=numeric(), Formula=character())

while (top_guardados < PARAM$GA$top_features && idx <= length(ordenados)) {
  i <- ordenados[idx]
  idx <- idx + 1

  if (is.na(scores_finales[i]) || scores_finales[i] <= 0.50) next
  if (is.na(formulas_finales[i]) || formulas_finales[i] %in% formulas_vistas) next
  if (es_expresion_trivial(formulas_finales[i])) next

  eval_res <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dataset),
    error = function(e) NULL
  )
  if (is.null(eval_res) || length(unique(eval_res[is.finite(eval_res)])) <= 1) next

  top_guardados <- top_guardados + 1
  formulas_vistas <- c(formulas_vistas, formulas_finales[i])

  nombre_col <- paste0("GA_Feature_", top_guardados)
  ga_cols_creadas <- c(ga_cols_creadas, nombre_col)

  cat(sprintf("[%s] AUC Univariado: %.5f | Fórmula: %s\n", nombre_col, scores_finales[i], formulas_finales[i]))
  dataset[, (nombre_col) := eval_res]
  dt_trazabilidad <- rbind(dt_trazabilidad, list(nombre_col, scores_finales[i], formulas_finales[i]))
}

fwrite(dt_trazabilidad, file = file.path(dir_experimento_base, paste0("GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv")), sep = ",")
cat("\nArchivo de trazabilidad guardado en: GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv\n")

cat("\nColumnas generadas por Algoritmo Genético e inyectadas al dataset:", paste(ga_cols_creadas, collapse = ", "), "\n")

# Blindaje anti-leakage: asegura que clase01 no quede en dataset antes de FEhist
if ("clase01" %in% colnames(dataset)) dataset[, clase01 := NULL]


In [ ]:
# ================================================================
# CONTROL GA - PROBLEMA #10 GRUPO A
# No modifica el dataset: verifica el resultado del bloque gramEvol.
# ================================================================

if (!exists("ga_cols_creadas"))
  stop("CONTROL GA: no existe ga_cols_creadas.")

if (length(ga_cols_creadas) == 0)
  stop("CONTROL GA: no se generó ninguna feature.")

if (anyDuplicated(ga_cols_creadas) > 0)
  stop("CONTROL GA: hay nombres de features duplicados.")

faltan_ga <- setdiff(ga_cols_creadas, colnames(dataset))
if (length(faltan_ga) > 0)
  stop("CONTROL GA: faltan columnas en dataset: ",
       paste(faltan_ga, collapse = ", "))

if ("clase01" %in% colnames(dataset))
  stop("CONTROL GA: clase01 quedó dentro del dataset antes de FEhist.")

if (length(ga_cols_creadas) < PARAM$GA$top_features) {
  warning(
    "CONTROL GA: se solicitaron ",
    PARAM$GA$top_features,
    " features, pero se generaron ",
    length(ga_cols_creadas),
    ". Revisar trazabilidad antes de continuar."
  )
}

cat(
  "CONTROL GA OK | candidatas:", length(candidatas_GA),
  "| features solicitadas:", PARAM$GA$top_features,
  "| features creadas:", length(ga_cols_creadas),
  "| columnas dataset antes de FEhist:", ncol(dataset),
  "\n"
)

print(dt_trazabilidad)


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)


#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest


#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# ================================================================
# 6.3.1.5 FEhist - Feature Engineering Histórico
# Recomendación elegida: PROBLEMA #05 - GRUPO B - Exp9112
#
# Variables incorporadas:
#   lag1 + lag2 + lag3
#   delta1 + delta2 + delta3
#   ma3
#
# Se aplican a todas las variables excepto:
# numero_de_cliente, foto_mes y clase_ternaria
# ================================================================


# Todo es lagueable, menos la primary key, el mes y la clase
cols_lagueables <- copy(setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
))


# ------------------------------------------------
# LAG 1
# valor de la variable un mes atrás
# ------------------------------------------------
dataset[,
    paste0(cols_lagueables, "_lag1") :=
        shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]


# ------------------------------------------------
# LAG 2
# valor de la variable dos meses atrás
# ------------------------------------------------
dataset[,
    paste0(cols_lagueables, "_lag2") :=
        shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]


# ------------------------------------------------
# LAG 3
# valor de la variable tres meses atrás
# ------------------------------------------------
dataset[,
    paste0(cols_lagueables, "_lag3") :=
        shift(.SD, 3, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]


# ------------------------------------------------
# DELTAS
# diferencia entre el valor actual y cada lag
# ------------------------------------------------
for (vcol in cols_lagueables)
{
    dataset[,
        paste0(vcol, "_delta1") :=
            get(vcol) - get(paste0(vcol, "_lag1"))
    ]

    dataset[,
        paste0(vcol, "_delta2") :=
            get(vcol) - get(paste0(vcol, "_lag2"))
    ]

    dataset[,
        paste0(vcol, "_delta3") :=
            get(vcol) - get(paste0(vcol, "_lag3"))
    ]
}


# ------------------------------------------------
# MEDIA MÓVIL DE 3 MESES
# mes actual + dos meses anteriores
# ------------------------------------------------
dataset[,
    paste0(cols_lagueables, "_ma3") :=
        frollmean(
            .SD,
            n = 3,
            align = "right",
            fill = NA,
            na.rm = TRUE
        ),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]


In [ ]:
# ================================================================
# CONTROL 2 - FE HISTÓRICO
# Verifica lag3, delta3 y ma3 sin modificar el dataset.
# ================================================================

lag3_esperados <- paste0(cols_lagueables, "_lag3")
delta3_esperados <- paste0(cols_lagueables, "_delta3")
ma3_esperados <- paste0(cols_lagueables, "_ma3")

faltan_lag3 <- setdiff(lag3_esperados, colnames(dataset))
faltan_delta3 <- setdiff(delta3_esperados, colnames(dataset))
faltan_ma3 <- setdiff(ma3_esperados, colnames(dataset))

if (length(faltan_lag3) > 0)
  stop("CONTROL FEhist: faltan columnas lag3.")

if (length(faltan_delta3) > 0)
  stop("CONTROL FEhist: faltan columnas delta3.")

if (length(faltan_ma3) > 0)
  stop("CONTROL FEhist: faltan columnas ma3.")

cat(
  "CONTROL FEhist OK | columnas base lagueables:", length(cols_lagueables),
  "| columnas totales después de FEhist:", ncol(dataset),
  "| tamaño dataset:", format(object.size(dataset), units = "auto"), "\n"
)


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)


#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos


### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> Para esta referencia final de la *Modalidad Gerencial* se utiliza PARAM$trainingstrategy$training_pct <- 1.00, es decir, 100% de la clase CONTINUA durante el Grid Search.
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

# Problema #12 - Grupo A (Víctor y David)
# Usamos 100% de CONTINUA durante el Grid Search
PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")


In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]


In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)


En el notebook original esta etapa incluye la instalación de LightGBM y se advierte una demora importante en Colab.
<br>En Cloud/Jupyter el tiempo dependerá de si `lightgbm` ya está instalado en la VM y de los recursos asignados.


In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)


In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)


In [ ]:
# ================================================================
# CONTROL 3 - TRAINING / UNDERSAMPLING
# Comprueba que entren todos los positivos y que el undersampling
# afecte únicamente a CONTINUA.
# ================================================================

control_training <- dataset[
  foto_mes %in% PARAM$trainingstrategy$training,
  .(
    filas_training = .N,
    positivos_total = sum(clase01 == 1L, na.rm = TRUE),
    positivos_usados = sum(fold_train & clase01 == 1L, na.rm = TRUE),
    continua_total = sum(clase01 == 0L, na.rm = TRUE),
    continua_usados = sum(fold_train & clase01 == 0L, na.rm = TRUE)
  )
]

control_training[
  , pct_continua_real := continua_usados / continua_total
]

print(control_training)

if (control_training$positivos_usados != control_training$positivos_total)
  stop("CONTROL Training: no están entrando todos los positivos.")

if (control_training$continua_usados <= 0)
  stop("CONTROL Training: no entró ningún CONTINUA.")

cat(
  "CONTROL Training OK | training_pct configurado:",
  PARAM$trainingstrategy$training_pct,
  "| proporción real CONTINUA:",
  round(control_training$pct_continua_real, 4), "\n"
)


####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}


seteo del Grid Search

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)


Corrida del Grid Search: aquí se hace el trabajo pesado.
<br>El notebook original advierte aproximadamente 50 minutos en Colab.
<br>En la VM Cloud conviene registrar el tiempo real de cada corrida porque dependerá de los recursos de la máquina.


In [ ]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]


la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)


In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros


### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en optimización de hiperparámetros

In [ ]:
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño


##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los mejores encontrados por el Grid Search
# FIX heredado de 7192: modifyList reemplaza correctamente los valores fijos
# por los hiperparametros ganadores del Grid Search.
param_final <- modifyList(
  fijos,
  PARAM$out$lgbm$mejores_hiperparametros
)


In [ ]:
# ================================================================
# CONTROL 4 - PARAM_FINAL CORREGIDO (7193, FIX heredado de 7192)
#
# Verifica que los hiperparametros elegidos por el Grid Search
# hayan reemplazado realmente a los valores fijos.
# ================================================================

mejores <- PARAM$out$lgbm$mejores_hiperparametros

# No deberían existir nombres duplicados al usar modifyList()
nombres_duplicados <- unique(names(param_final)[duplicated(names(param_final))])

if (length(nombres_duplicados) > 0) {
  stop(
    "CONTROL param_final 7193: todavía hay nombres duplicados: ",
    paste(nombres_duplicados, collapse = ", ")
  )
}

# Verifico uno por uno los hiperparametros elegidos por el Grid Search
for (nm in names(mejores)) {

  valor_final <- param_final[[nm]]
  valor_grid  <- mejores[[nm]]

  if (!isTRUE(all.equal(valor_final, valor_grid))) {
    stop(
      "CONTROL param_final 7193: ",
      nm,
      " no coincide. Grid Search=",
      valor_grid,
      " | param_final=",
      valor_final
    )
  }
}

cat(
  "CONTROL param_final 7193 OK | ",
  "num_leaves:", param_final$num_leaves,
  "| min_data_in_leaf:", param_final$min_data_in_leaf,
  "| num_iterations:", param_final$num_iterations,
  "\n"
)


##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)


In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")


In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)


#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]


In [ ]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)


##### Tabla Prediccion

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)


#### Kaggle Competition Submit

### PASO 2 — Generar CSV y, solo cuando se decida, enviarlos a Kaggle

Al hacer **Run All**, la celda siguiente queda en:

```r
PARAM$control$submit_kaggle <- FALSE
```

Con `FALSE`:

- controla las predicciones;
- genera los 11 CSV (`800, 850, ..., 1300`);
- **NO envía nada a Kaggle**.

Si el **PASO 1** mostró `CONTROL KAGGLE OK` y toda la corrida terminó correctamente,
para hacer los submits:

1. cambiar **solo** esta línea dentro de la celda siguiente:

```r
PARAM$control$submit_kaggle <- TRUE
```

2. ejecutar **solamente la celda PASO 2**.

No hace falta volver a ejecutar FE, GA, FE histórico, Grid Search ni Final Training.


In [ ]:
# ================================================================
# PASO 2 - KAGGLE
#
# RUN ALL SEGURO:
#   FALSE = genera los 11 CSV y NO envía nada.
#
# PARA SUBIR A KAGGLE:
#   1) cambiar SOLO la siguiente línea de FALSE a TRUE
#   2) ejecutar SOLAMENTE ESTA CELDA
# ================================================================

PARAM$control$submit_kaggle <- TRUE   # <-- CAMBIAR A TRUE SOLO PARA SUBIR

PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

# ------------------------------------------------
# A) Controles previos de las predicciones
# ------------------------------------------------
if (!exists("tb_prediccion"))
  stop("CONTROL Kaggle: no existe tb_prediccion.")

if (nrow(tb_prediccion) < max(PARAM$kaggle$cortes))
  stop("CONTROL Kaggle: hay menos filas que el corte máximo.")

if (anyDuplicated(tb_prediccion$numero_de_cliente) > 0)
  stop("CONTROL Kaggle: hay numero_de_cliente duplicados.")

if (any(!is.finite(tb_prediccion$prob)))
  stop("CONTROL Kaggle: hay probabilidades NA/Inf/NaN.")

cat(
  "CONTROL PRE-KAGGLE OK | clientes:", nrow(tb_prediccion),
  "| prob min:", min(tb_prediccion$prob),
  "| prob max:", max(tb_prediccion$prob), "\n"
)

setorder(tb_prediccion, -prob)
dir.create("kaggle", showWarnings = FALSE)

# ------------------------------------------------
# B) Genero SIEMPRE los 11 CSV
# ------------------------------------------------
for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0(
    "./kaggle/KA", PARAM$experimento, "_", envios, ".csv"
  )

  fwrite(
    tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )

  cat("CSV generado:", archivo_kaggle, "\n")
}

# ------------------------------------------------
# C) MODO SEGURO
# ------------------------------------------------
if (!isTRUE(PARAM$control$submit_kaggle)) {

  cat("\n============================================================\n")
  cat("PASO 2 EN MODO CONTROL: NO SE REALIZÓ NINGÚN SUBMIT\n")
  cat("Los 11 CSV quedaron generados en ./kaggle/\n")
  cat("\nPara subirlos:\n")
  cat("1) cambiar PARAM$control$submit_kaggle <- FALSE por TRUE\n")
  cat("2) volver a ejecutar SOLAMENTE esta celda PASO 2\n")
  cat("============================================================\n")

} else {

  # ------------------------------------------------
  # D) Reconfirmo acceso antes de enviar
  # ------------------------------------------------
  kaggle_cmd <- Sys.which("kaggle")

  if (nchar(kaggle_cmd) == 0)
    stop("SUBMIT Kaggle: no se encontró la CLI de kaggle.")

  salida_acceso <- suppressWarnings(
    system2(
      kaggle_cmd,
      args = c(
        "competitions", "submissions",
        "-c", PARAM$kaggle$competencia,
        "--page-size", "1",
        "--csv",
        "--quiet"
      ),
      stdout = TRUE,
      stderr = TRUE
    )
  )

  estado_acceso <- attr(salida_acceso, "status")
  if (is.null(estado_acceso)) estado_acceso <- 0

  if (estado_acceso != 0) {
    cat(paste(salida_acceso, collapse = "\n"), "\n")
    stop(
      "SUBMIT Kaggle: no hay acceso a ",
      PARAM$kaggle$competencia,
      ". No se envió ningún archivo."
    )
  }

  cat("\nACCESO KAGGLE OK. INICIO DE LOS 11 SUBMITS.\n")

  # ------------------------------------------------
  # E) Envío real; si uno falla, se detiene
  # ------------------------------------------------
  for (i in seq_along(PARAM$kaggle$cortes)) {

    envios <- PARAM$kaggle$cortes[i]

    archivo_kaggle <- paste0(
      "./kaggle/KA", PARAM$experimento, "_", envios, ".csv"
    )

    mensaje <- paste0(
      "envios=", envios,
      " semilla=", PARAM$semilla_primigenia
    )

    cat("\nSUBMIT:", archivo_kaggle, "\n")

    salida_submit <- suppressWarnings(
      system2(
        kaggle_cmd,
        args = c(
          "competitions", "submit",
          "-c", PARAM$kaggle$competencia,
          "-f", archivo_kaggle,
          "-m", shQuote(mensaje)
        ),
        stdout = TRUE,
        stderr = TRUE
      )
    )

    estado_submit <- attr(salida_submit, "status")
    if (is.null(estado_submit)) estado_submit <- 0

    cat(paste(salida_submit, collapse = "\n"), "\n")

    if (estado_submit != 0) {
      stop(
        "SUBMIT Kaggle FALLÓ en el corte ", envios,
        ". Se detiene el proceso para no seguir enviando."
      )
    }

    if (i < length(PARAM$kaggle$cortes))
      Sys.sleep(30)
  }

  cat("\n============================================================\n")
  cat("SUBMITS FINALIZADOS CORRECTAMENTE\n")
  cat("Experimento:", PARAM$experimento, "\n")
  cat("Competencia:", PARAM$kaggle$competencia, "\n")
  cat("Cantidad de envíos:", length(PARAM$kaggle$cortes), "\n")
  cat("============================================================\n")
}


In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
